# Configuration

In [37]:
library(readr)
library(GRaNIE)
library(Seurat)
library(Signac)
library(cowplot)
library(data.table)
library(ggplot2)
library(patchwork)
library(stringr)
library(Matrix)
library(org.Hs.eg.db) # 人类数据
library(AnnotationDbi)
library(dplyr)
library(tibble)


# Data read

In [38]:


cell_type <- "HepG2"

data_path <- paste0("/home/liyang/BioWuYan/dygmamba_project/data/cell_line/", cell_type, "/process/")


# The only need input are the atac_dir and rna_dir

atac_dir <- paste0(data_path,  "R/atac/")

rna_dir <- paste0(data_path,  "R/rna/")

############################################################################################
#******************************** Data Read --- ATAC ***************************************
############################################################################################

mtx <- readMM(paste0(atac_dir, "sparse.mtx"))

cellinfo <- fread(paste0(atac_dir, "cellinfo.csv"), header = T, data.table = F)

atacinfo <- fread(paste0(atac_dir, "atacinfo.csv"), header = T, data.table = F)

rownames(mtx) <- cellinfo$V1
colnames(mtx) <- atacinfo$V1

meta <- cellinfo
rownames(meta) <- cellinfo$V1

if(identical(rownames(meta), rownames(mtx))){
        # 行需为 feature
    print("Info identical, Can create Seurat Object")
    mtx_transposed <- t(mtx) 

    atac_seurat <- CreateSeuratObject(count = mtx_transposed, meta.data = meta, assay = "ATAC")
}

rownames(atac_seurat) <- sub("-", ":", rownames(atac_seurat)) # 确保只有第一个是冒号

############################################################################################
#******************************** Data Read --- RNA ***************************************
############################################################################################


rna_mtx <- readMM(paste0(rna_dir, "sparse.mtx"))

rna_cellinfo <- fread(paste0(rna_dir, "cellinfo.csv"), header = T, data.table = F)

rna_info <- fread(paste0(rna_dir, "rnainfo.csv"), header = T, data.table = F)

rownames(rna_mtx) <- rna_cellinfo$V1
colnames(rna_mtx) <- rna_info$Geneid

meta <- rna_cellinfo
rownames(meta) <- rna_cellinfo$V1


    
print("Create Seurat Object")
rna_mtx_transposed <- t(rna_mtx)

rna_seurat <- CreateSeuratObject(count = rna_mtx_transposed, meta.data = meta, assay = "RNA")



[1] "Info identical, Can create Seurat Object"


Warning message:
"Data is of class dgTMatrix. Coercing to dgCMatrix."


[1] "Create Seurat Object"


Warning message:
"Data is of class dgTMatrix. Coercing to dgCMatrix."


# Data process

In [39]:

n_bins <- 50 

rna_seurat <- NormalizeData(rna_seurat)
rna_seurat <- FindVariableFeatures(rna_seurat)
rna_seurat <- ScaleData(rna_seurat)
rna_seurat <- RunPCA(rna_seurat, verbose = FALSE) # 生成 PCA 降维

rna_seurat <- RunUMAP(rna_seurat, dims = 1:30)
rna_kmeans <- kmeans(Embeddings(rna_seurat, "umap"), centers = n_bins)

rna_seurat$metacell_id <- paste0("bin_", rna_kmeans$cluster)


atac_seurat <- RunTFIDF(atac_seurat)
atac_seurat <- FindTopFeatures(atac_seurat, min.cutoff = 'q0')
atac_seurat <- RunSVD(atac_seurat) # ATAC 的降维通常叫 LSI 或 SVD

# 3. 运行 UMAP (基于 lsi)
atac_seurat <- RunUMAP(atac_seurat, reduction = 'lsi', dims = 2:30)

atac_kmeans <- kmeans(Embeddings(atac_seurat, "umap"), centers = n_bins)
atac_seurat$metacell_id <- paste0("bin_", atac_kmeans$cluster)





Normalizing layer: counts



Finding variable features for layer counts

Centering and scaling data matrix

14:02:31 UMAP embedding parameters a = 0.9922 b = 1.112

14:02:31 Read 500 rows and found 30 numeric columns

14:02:31 Using Annoy for neighbor search, n_neighbors = 30

14:02:31 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----|

*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
|

14:02:31 Writing NN index file to temp file /tmp/RtmpihyBYP/file18da01c0801c8

14:02:31 Searching Annoy index using 1 thread, search_k = 3000

14:02:31 Annoy recall = 100%

14:02:33 Commencing smooth kNN distance calibration using 1 thread
 with target n_neighbors = 30

14:02:35 Initializing from normalized Laplacian + noise (using RSpectra)

14:02:35 Commencing optimization for 500 epochs, with 15574 positive edges

14:02:35 Using rng type: pcg

14:02:37 Optimization 

In [40]:
# --- 处理 RNA ---
DefaultAssay(rna_seurat) <- "RNA" # 显式设置
# 聚合
rna_assay_name <- DefaultAssay(rna_seurat)
pb_list_rna <- AggregateExpression(rna_seurat, group.by = "metacell_id", 
                                   assays = rna_assay_name, slot = "counts")
pb_rna <- pb_list_rna[[rna_assay_name]] # 动态提取

# --- 处理 ATAC ---
atac_assay_name <- DefaultAssay(atac_seurat)
pb_list_atac <- AggregateExpression(atac_seurat, group.by = "metacell_id", 
                                    assays = atac_assay_name, slot = "counts")
pb_atac <- pb_list_atac[[atac_assay_name]]


# --- 对齐数据 ---
common_bins <- intersect(colnames(pb_rna), colnames(pb_atac))
pb_rna <- pb_rna[, common_bins]
pb_atac <- pb_atac[, common_bins]


# GRN inference

In [41]:

# 1. 修正 ATAC 矩阵：将行名提取为 "peakID" 列
pb_atac_df <- as.data.frame(pb_atac) %>% 
  rownames_to_column(var = "peakID")

# 2. 修正 RNA 矩阵：将行名提取为 "geneID" 列
pb_rna_df <- as.data.frame(pb_rna) %>% 
  rownames_to_column(var = "geneID")

# 检查一下现在长什么样 (你应该能看到第一列变成了 ID)
print(head(pb_atac_df[, 1:5])) 

metadata_df <- data.frame(sample_id = common_bins, cell_type = "SingleType")

GRN <- initializeGRN(objectMetadata = list(name = "Metacell_GRN"), 
                      genomeAssembly = "hg38", 
                      outputFolder = "GRaNIE_Results")


                    peakID bin-1 bin-10 bin-11 bin-12
1 chr1:100037630-100038936     7      0      2      6
2 chr1:100132442-100133739     7      0      0      2
3     chr1:1001683-1001983     2      4      1      4
4 chr1:100249267-100250230     0      2      4      3
5 chr1:100265726-100266824     2      6      0      2
6 chr1:100320471-100321812     0      0      2      2


INFO [2026-02-02 14:02:57] Empty GRN object created successfully. Type the object name (e.g., GRN) to retrieve summary information about it at any time.

INFO [2026-02-02 14:02:57]  Default output folder: /home/liyang/BioWuYan/dygmamba_project/model/GRaNIE/code/GRaNIE_Results/

INFO [2026-02-02 14:02:57]  Genome assembly: hg38

INFO [2026-02-02 14:02:57] Finished successfully. Execution time: 0 secs



In [42]:

current_symbols <- pb_rna_df$geneID # 假设你之前的 ID 列叫 geneID
ensembl_ids <- mapIds(org.Hs.eg.db,
                      keys = current_symbols,
                      column = "ENSEMBL",
                      keytype = "SYMBOL",
                      multiVals = "first")


pb_rna_df_fixed <- pb_rna_df %>%
            mutate(ensemblID = ensembl_ids[geneID]) %>% # 根据旧 ID 匹配新 ID
            dplyr::filter(!is.na(ensemblID)) %>%               # 去掉没找到 Ensembl ID 的基因
            dplyr::distinct(ensemblID, .keep_all = TRUE)       # 去掉重复的 ID (非常重要！)

pb_rna_clean <- pb_rna_df_fixed %>%
            dplyr::select(ensemblID, where(is.numeric)) 


GRN <- addData(GRN, 
               counts_peaks = pb_atac_df,      # ATAC 数据不变
               counts_rna = pb_rna_clean,   # 使用修复后的 RNA 数据
               sampleMetadata = metadata_df,   # 元数据不变
               idColumn_peaks = "peakID",      # ATAC 的 ID 列名
               idColumn_RNA = "ensemblID",     # ⚠️ 注意：这里改成新的列名
               force = TRUE)

'select()' returned 1:many mapping between keys and columns

INFO [2026-02-02 14:02:58]  Normalizing data using the package DESeq2 with a standard size factor normalization.

INFO [2026-02-02 14:02:58]  Normalizing data using the package limma with the following method: limma_quantile

INFO [2026-02-02 14:02:59] Parsing provided metadata...

INFO [2026-02-02 14:02:59] Subset RNA and peaks and keep only shared samples

INFO [2026-02-02 14:02:59]  Number of samples for RNA before filtering: 50

INFO [2026-02-02 14:02:59]  Number of samples for peaks before filtering: 50

INFO [2026-02-02 14:02:59]  50 samples (bin-1,bin-10,bin-11,bin-12,bin-13,bin-14,bin-15,bin-16,bin-17,bin-18,bin-19,bin-2,bin-20,bin-21,bin-22,bin-23,bin-24,bin-25,bin-26,bin-27,bin-28,bin-29,bin-3,bin-30,bin-31,bin-32,bin-33,bin-34,bin-35,bin-36,bin-37,bin-38,bin-39,bin-4,bin-40,bin-41,bin-42,bin-43,bin-44,bin-45,bin-46,bin-47,bin-48,bin-49,bin-5,bin-50,bin-6,bin-7,bin-8,bin-9) are shared between the peaks and RNA-Seq d

In [43]:
# 1. 从本地数据库提取 Symbol 到 Ensembl 的对照表
keys <- keys(org.Hs.eg.db, keytype = "SYMBOL")
mapping <- select(org.Hs.eg.db, 
                  keys = keys, 
                  columns = c("ENSEMBL", "SYMBOL"), 
                  keytype = "SYMBOL")

# 2. 整理成 GRaNIE 格式 (使用了 dplyr:: 前缀来防止报错)
translation_table_local <- mapping %>%
                  dplyr::rename(TF_name = SYMBOL, TF_ensembl = ENSEMBL) %>%
                  dplyr::filter(!is.na(TF_ensembl)) %>%        # <--- 修正点：加上 dplyr::
                  dplyr::distinct(TF_name, .keep_all = TRUE)   # <--- 修正点：加上 dplyr::

print(paste("本地构建了", nrow(translation_table_local), "个基因的对照表"))

# 使用本地表运行 addTFBS
GRN <- addTFBS(GRN, 
               source = "JASPAR2024", 
               translationTable = translation_table_local)



# 2. 计算 TFBS 与 Peak 的重叠
# 这一步现在会使用 JASPAR 的 Motif 进行扫描
GRN <- overlapPeaksAndTFBS(GRN)


# 3. 计算 TF-Peak 链接
GRN <- addConnections_TF_peak(GRN, 
                              corMethod = "pearson")

# 4. 计算 Peak-Gene 链接
GRN <- addConnections_peak_gene(GRN,  
                                corMethod = "pearson")


'select()' returned 1:many mapping between keys and columns



[1] "本地构建了 37765 个基因的对照表"


INFO [2026-02-02 14:03:13] Querying JASPAR2024 database. This may take a while.

Retrieving a list of species available in JASPAR release 2022.

INFO [2026-02-02 14:03:26] Retrieving gene annotation for JASPAR TFs. This may take a while.

INFO [2026-02-02 14:03:30]  Retrieving BioMart database succeeded

INFO [2026-02-02 14:03:31]  Retrieving genome annotation succeeded

INFO [2026-02-02 14:03:31]  TF statistics:

INFO [2026-02-02 14:03:31]   Number of TFs as returned by JASPAR: 720

INFO [2026-02-02 14:03:31]   Number of TFs with a unique mapping to an Ensembl ID: 619

INFO [2026-02-02 14:03:31]   Number of TFs with a non-unique mapping to an Ensembl ID: 33 (MA0153.2, MA0468.1, MA0482.3, MA0486.2, MA0509.3, MA0630.2, MA0653.1, MA0657.2, MA0715.1, MA0736.1, MA0743.3, MA0749.2, MA0772.2, MA0773.1, MA0821.2, MA0831.3, MA0842.3, MA0849.1, MA0855.1, MA0864.3, MA0879.3, MA1113.3, MA1115.2, MA1421.1, MA1525.3, MA1555.1, MA1583.2, MA1649.2, MA1715.1, MA1720.2, MA1723.2, MA2333.1, MA2335.1)

I

# Analysis

In [ ]:
GRN <- filterGRNAndConnectGenes(GRN, 
                                TF_peak.fdr.threshold = 1, 
                                peak_gene.fdr.threshold = 1)

results_df <- getGRNConnections(GRN)

write.csv(results_df, paste0(data_path, "GRaNIE_Full_Links.csv"), row.names = FALSE)
# # tf_region_df <- results_df %>%
# #   dplyr::select(TF.name, TF.ENSEMBL, peak.ID, TF_peak.r, TF_peak.fdr) %>%
# #   dplyr::distinct() # 去重，因为一个 TF-Peak 组合可能对应多个靶基因

# # head(tf_region_df)
# GRN

INFO [2026-02-02 14:05:30] Filter GRN network

INFO [2026-02-02 14:05:30] 

real data

INFO [2026-02-02 14:05:30] Inital number of rows left before all filtering steps: 152

INFO [2026-02-02 14:05:30]  Filter network and retain only rows with TF-peak connections with an FDR < 1

INFO [2026-02-02 14:05:30]   Number of TF-peak rows before filtering TFs: 152

INFO [2026-02-02 14:05:30]   Number of TF-peak rows after filtering TFs: 152

INFO [2026-02-02 14:05:30] 2. Filter peak-gene connections

INFO [2026-02-02 14:05:30] 3. Merging TF-peak with peak-gene connections and filter the combined table...

INFO [2026-02-02 14:05:30] Inital number of rows left before filtering steps: 257

INFO [2026-02-02 14:05:30]  Filter TF-TF self-loops

INFO [2026-02-02 14:05:30]   Number of rows before filtering genes: 257

INFO [2026-02-02 14:05:30]   Number of rows after filtering genes: 250

INFO [2026-02-02 14:05:30]  Filter network and retain only rows with peak_gene.r in the following interval: (0 - 1]

INFO [2026-02-02 14:05:30]   Number of rows before filtering genes (including/excluding NA): 128/127

INFO [2026-02-02 14:05:30]   Number of rows after filtering genes (including/excluding NA): 128/127

INFO [2026-02-02 14:05:30] Final number of rows left after all filtering steps: 128

INFO [2026-02-02 14:05:30] 

permuted data

INFO [2026-02-02 14:05:30] Inital number of rows left before all filtering steps: 70

INFO [2026-02-02 14:05:30]  Filter network and retain only rows with TF-peak connections with an FDR < 1

INFO [2026-02-02 14:05:30]   Number of TF-peak rows before filtering TFs: 70

INFO [2026-02-02 14:05:30]   Number of TF-peak rows after filtering TFs: 70

INFO [2026-02-02 14:05:30] 2. Filter peak-gene connections

INFO [2026-02-02 14:05:30] 3. Merging TF-peak with peak-gene connections and filter the combined table...

INFO [2026-02-02 14:05:30] Inital number of rows left before filtering steps: 129

INFO [2026-02-02 14:05:30]  Filter TF-TF self-loops

INFO [2026-02-02 1

TF.ID,TF.name,TF.ENSEMBL,peak.ID,TF_peak.r_bin,TF_peak.r,TF_peak.fdr,TF_peak.fdr_direction,TF_peak.connectionType,peak_gene.source,peak_gene.distance,peak_gene.r,peak_gene.p_raw,peak_gene.p_adj,gene.ENSEMBL,gene.name,gene.type
<fct>,<fct>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<int>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>
MA0101.1,REL,ENSG00000162924,chr1:37793478-37794775,"(-0.45,-0.4]",-0.4023783,0.28824763,neg,expression,neighborhood,0,0.02446782,0.8660619,0.9532171,ENSG00000185090,MANEAL,protein_coding
MA0101.1,REL,ENSG00000162924,chr12:132955270-132956649,"(-0.4,-0.35]",-0.3811631,0.29021487,neg,expression,neighborhood,29716,0.08231550,0.5698296,0.9532171,ENSG00000198393,ZNF26,protein_coding
MA0101.1,REL,ENSG00000162924,chr15:90249266-90249826,"(-0.4,-0.35]",-0.3561285,0.29021487,neg,expression,neighborhood,138416,0.13259588,0.3586492,0.9432333,ENSG00000140575,IQGAP1,protein_coding
MA0101.1,REL,ENSG00000162924,chr16:27624867-27625626,"(-0.4,-0.35]",-0.3583290,0.29021487,neg,expression,neighborhood,165313,0.19077131,0.1844878,0.9432333,ENSG00000077235,GTF3C1,protein_coding
MA0101.1,REL,ENSG00000162924,chr16:85798374-85800492,"(-0.4,-0.35]",-0.3550653,0.29021487,neg,expression,neighborhood,0,0.06967765,0.6306453,0.9532171,ENSG00000131143,COX4I1,protein_coding
MA0101.1,REL,ENSG00000162924,chr17:75876194-75878942,"(-0.4,-0.35]",-0.3508926,0.29021487,neg,expression,neighborhood,1393,0.20409447,0.1551197,0.9432333,ENSG00000141569,TRIM65,protein_coding
MA0101.1,REL,ENSG00000162924,chr2:219176329-219177316,"(-0.4,-0.35]",-0.3610473,0.29021487,neg,expression,neighborhood,4433,0.02515632,0.8623290,0.9532171,ENSG00000115649,CNPPD1,protein_coding
MA0101.1,REL,ENSG00000162924,chr2:219176329-219177316,"(-0.4,-0.35]",-0.3610473,0.29021487,neg,expression,neighborhood,105,0.15111923,0.2948331,0.9432333,ENSG00000144567,RETREG2,protein_coding
MA0101.1,REL,ENSG00000162924,chr2:72886978-72889172,"(-0.4,-0.35]",-0.3575796,0.29021487,neg,expression,neighborhood,52864,0.01984034,0.8912226,0.9532171,ENSG00000144040,SFXN5,protein_coding


In [49]:
# # 5. 过滤并生成网络
# # 对于 Metacell 数据，建议先用 0.2 的 FDR 试试水，如果结果太多再收紧到 0.1
# GRN <- filterGRNAndConnectGenes(GRN, TF_peak.fdr.threshold = 0.1,
#                                 peak_gene.fdr.threshold = 0.1)


# # 6. 导出结果
# results_df <- getGRNConnections(GRN, type = "all.filtered")
# # write.csv(results_df, "GRaNIE_Final_Network.csv", row.names = FALSE)

# # 打印结果概览
# print(paste("找到的连接数:", nrow(results_df)))
rownames(results_df)

[1] "1"   "2"   "3"   "4"   "5"   "6"   "7"   "8"   "9"   "10"  "11"  "12" 
 [13] "13"  "14"  "15"  "16"  "17"  "18"  "19"  "20"  "21"  "22"  "23"  "24" 
 [25] "25"  "26"  "27"  "28"  "29"  "30"  "31"  "32"  "33"  "34"  "35"  "36" 
 [37] "37"  "38"  "39"  "40"  "41"  "42"  "43"  "44"  "45"  "46"  "47"  "48" 
 [49] "49"  "50"  "51"  "52"  "53"  "54"  "55"  "56"  "57"  "58"  "59"  "60" 
 [61] "61"  "62"  "63"  "64"  "65"  "66"  "67"  "68"  "69"  "70"  "71"  "72" 
 [73] "73"  "74"  "75"  "76"  "77"  "78"  "79"  "80"  "81"  "82"  "83"  "84" 
 [85] "85"  "86"  "87"  "88"  "89"  "90"  "91"  "92"  "93"  "94"  "95"  "96" 
 [97] "97"  "98"  "99"  "100" "101" "102" "103" "104" "105" "106" "107" "108"
[109] "109" "110" "111" "112" "113" "114" "115" "116" "117" "118" "119" "120"
[121] "121" "122" "123" "124" "125" "126" "127" "128"

# Test

## TF-region

In [53]:
tf_region_df <- results_df %>%
  dplyr::select(TF.name, TF.ENSEMBL, peak.ID, TF_peak.r, TF_peak.fdr) %>%
  dplyr::distinct() # 去重，因为一个 TF-Peak 组合可能对应多个靶基因

tf_region_df
write.csv(tf_region_df, paste0(data_path, "GRaNIE_tf_region_Links.csv"), row.names = FALSE)

TF.name,TF.ENSEMBL,peak.ID,TF_peak.r,TF_peak.fdr
<fct>,<fct>,<fct>,<dbl>,<dbl>
REL,ENSG00000162924,chr1:37793478-37794775,-0.4023783,0.28824763
REL,ENSG00000162924,chr12:132955270-132956649,-0.3811631,0.29021487
REL,ENSG00000162924,chr15:90249266-90249826,-0.3561285,0.29021487
REL,ENSG00000162924,chr16:27624867-27625626,-0.3583290,0.29021487
REL,ENSG00000162924,chr16:85798374-85800492,-0.3550653,0.29021487
REL,ENSG00000162924,chr17:75876194-75878942,-0.3508926,0.29021487
REL,ENSG00000162924,chr2:219176329-219177316,-0.3610473,0.29021487
REL,ENSG00000162924,chr2:72886978-72889172,-0.3575796,0.29021487
REL,ENSG00000162924,chr7:6385306-6386140,-0.3804662,0.29021487


## Region-Gene

In [51]:
region_gene_df <- results_df %>%
  dplyr::select(peak.ID, gene.name, gene.ENSEMBL, peak_gene.r, peak_gene.p_adj, peak_gene.distance) %>%
  dplyr::distinct() # 去重，因为一个 Peak-Gene 组合可能被多个 TF 结合

write.csv(region_gene_df, paste0(data_path, "GRaNIE_region_gene_Links.csv"), row.names = FALSE)

peak.ID,gene.name,gene.ENSEMBL,peak_gene.r,peak_gene.p_adj,peak_gene.distance
<fct>,<fct>,<fct>,<dbl>,<dbl>,<int>
chr1:37793478-37794775,MANEAL,ENSG00000185090,0.02446782,0.9532171,0
chr12:132955270-132956649,ZNF26,ENSG00000198393,0.08231550,0.9532171,29716
chr15:90249266-90249826,IQGAP1,ENSG00000140575,0.13259588,0.9432333,138416
chr16:27624867-27625626,GTF3C1,ENSG00000077235,0.19077131,0.9432333,165313
chr16:85798374-85800492,COX4I1,ENSG00000131143,0.06967765,0.9532171,0
chr17:75876194-75878942,TRIM65,ENSG00000141569,0.20409447,0.9432333,1393


## TF-Gene

In [52]:
tf_gene_df <- results_df %>%
  dplyr::select(TF.name, gene.name, peak.ID, TF_peak.r, peak_gene.r)

write.csv(tf_gene_df, paste0(data_path, "GRaNIE_tf_gene_Links.csv"), row.names = FALSE)

TF.name,gene.name,peak.ID,TF_peak.r,peak_gene.r
<fct>,<fct>,<fct>,<dbl>,<dbl>
REL,MANEAL,chr1:37793478-37794775,-0.4023783,0.02446782
REL,ZNF26,chr12:132955270-132956649,-0.3811631,0.08231550
REL,IQGAP1,chr15:90249266-90249826,-0.3561285,0.13259588
REL,GTF3C1,chr16:27624867-27625626,-0.3583290,0.19077131
REL,COX4I1,chr16:85798374-85800492,-0.3550653,0.06967765
REL,TRIM65,chr17:75876194-75878942,-0.3508926,0.20409447
